Install and Imports

In [1]:
!pip -q install -U pandas numpy spacy tqdm
!python -m spacy download en_core_web_sm -q
import re, string, ast
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

import spacy
from spacy.matcher import PhraseMatcher
from spacy.lang.en.stop_words import STOP_WORDS

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Load annotated CSV

In [2]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/CS685/linkedin"
ANNOT_PATH = f"{BASE_DIR}/sentences_annotated_clean.csv"
ESCO_PATH  = "/content/drive/MyDrive/CS685/esco/skills_en.csv"

df = pd.read_csv(ANNOT_PATH)

print(df.shape)
print(df.columns)
df.head(3)

Mounted at /content/drive
(700, 6)
Index(['job_id', 'domain', 'sent_id', 'sentence', 'has_skill', 'spans'], dtype='object')


,job_id,domain,sent_id,sentence,has_skill,spans
0,3904978968,DA,3904978968_23,Excellent communication and collaboration skil...,True,communication and collaboration skills
1,3904568250,DA,3904568250_5,"In this role, you will harness your business a...",False,NaN
2,3899520919,DA,3899520919_1,"QualificationsAnalytical Skills, Data Analytic...",True,Analytical Skills; Data Analytics; Statistics;...


Parse gold spans into a list

In [3]:
def parse_gold_spans(raw):
    if pd.isna(raw):
        return []
    s = str(raw).strip()
    if not s:
        return []
    # split on semicolon
    parts = [p.strip() for p in s.split(";")]
    return [p for p in parts if p]

df["gold_spans"] = df["spans"].apply(parse_gold_spans)

# sanity
df[["sentence","has_skill","gold_spans"]].head(10)

,sentence,has_skill,gold_spans
0,Excellent communication and collaboration skil...,True,[communication and collaboration skills]
1,"In this role, you will harness your business a...",False,[]
2,"QualificationsAnalytical Skills, Data Analytic...",True,"[Analytical Skills, Data Analytics, Statistics..."
3,The Business Analyst will be responsible for t...,False,[]
4,"In this role, you will be responsible for desi...",True,[Drupal-based websites and applications]
5,Ability to quickly understand health and human...,False,[]
6,As a Senior Cloud Operations Developer you wil...,True,"[Cloud solutions, release and deployment proce..."
7,Minimum five years of hands-on experience usin...,True,"[Microsoft Office Suite, UML, flow charting so..."
8,Experience with machine learning frameworks (e.g.,True,[machine learning frameworks]
9,Experience with financial standards like ISO 8...,True,"[financial standards, ISO 8583, ACH/NACHA]"


Load ESCO + build variant dictionary

In [4]:
esco = pd.read_csv(ESCO_PATH)
print(esco.shape)
esco[["preferredLabel","altLabels","conceptUri"]].head(3)

(13939, 13)


,preferredLabel,altLabels,conceptUri
0,manage musical staff,manage staff of music\ncoordinate duties of mu...,http://data.europa.eu/esco/skill/0005c151-5b5a...
1,supervise correctional procedures,oversee prison procedures\nmanage correctional...,http://data.europa.eu/esco/skill/00064735-8fad...
2,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,http://data.europa.eu/esco/skill/000709ed-2be5...


In [5]:
PREF_COL = "preferredLabel"
ALT_COL  = "altLabels"
ID_COL   = "conceptUri"

def parse_alt_labels(raw):
    """ESCO altLabels are often newline-separated. Split safely."""
    if pd.isna(raw):
        return []
    text = str(raw)
    # split on newlines first
    items = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            items.append(line)
    # de-dup preserving order
    seen = set()
    out = []
    for x in items:
        x2 = x.strip()
        if x2 and x2 not in seen:
            seen.add(x2)
            out.append(x2)
    return out

def is_bad_variant(v: str) -> bool:
    v0 = v.strip()
    if len(v0) < 2:
        return True
    low = v0.lower()
    if low in STOP_WORDS:
        return True
    # purely punctuation
    if all(ch in string.punctuation for ch in low):
        return True
    # avoid weird single chars
    if len(low) == 1:
        return True
    return False

esco["alt_list"] = esco[ALT_COL].apply(parse_alt_labels)

rows = []
for _, r in esco.iterrows():
    canonical = str(r[PREF_COL]).strip()
    skill_id  = r[ID_COL]
    variants = [canonical] + r["alt_list"]
    for v in variants:
        v = str(v).strip()
        if not v or is_bad_variant(v):
            continue
        rows.append({
            "skill_id": skill_id,
            "canonical": canonical,
            "variant": v.lower()
        })

skill_dict_df = pd.DataFrame(rows).drop_duplicates(subset=["variant"])
print("Variants:", len(skill_dict_df))
skill_dict_df.head()

Variants: 99181


,skill_id,canonical,variant
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,manage musical staff
1,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,manage staff of music
2,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,coordinate duties of musical staff
3,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,manage music staff
4,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,direct musical staff


Build spaCy PhraseMatcher baseline (sentence-level)

In [6]:
nlp = spacy.load("en_core_web_sm", disable=["ner","parser","tagger","lemmatizer"])
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

variants = skill_dict_df["variant"].tolist()
patterns = [nlp.make_doc(v) for v in variants if v.strip()]
matcher.add("ESCO_SKILL", patterns)

# quick check
len(patterns)

99181

In [15]:
def baseline_extract_spans_raw(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    matches = matcher(doc)

    spans = []
    for _, start, end in matches:
        sp = doc[start:end]
        spans.append(sp.text)

    # de-dup (case-insensitive) while preserving order
    seen = set()
    out = []
    for s in spans:
        key = s.strip().lower()
        if key and key not in seen:
            seen.add(key)
            out.append(s.strip())
    return out

# run on your dataset
tqdm.pandas()
df["baseline_spans_raw"] = df["sentence"].progress_apply(baseline_extract_spans)

df[["sentence","gold_spans","baseline_spans"]].head(10)

100%|██████████| 700/700 [00:04<00:00, 170.22it/s]


,sentence,gold_spans,baseline_spans
0,Excellent communication and collaboration skil...,[communication and collaboration skills],[communication]
1,"In this role, you will harness your business a...",[],"[conduct, data, data analysis]"
2,"QualificationsAnalytical Skills, Data Analytic...","[Analytical Skills, Data Analytics, Statistics...","[Data, Data Analytics, communication, statisti..."
3,The Business Analyst will be responsible for t...,[],[be responsible]
4,"In this role, you will be responsible for desi...",[Drupal-based websites and applications],"[be responsible, Drupal]"
5,Ability to quickly understand health and human...,[],"[business processes, processes]"
6,As a Senior Cloud Operations Developer you wil...,"[Cloud solutions, release and deployment proce...","[processes, security, scale]"
7,Minimum five years of hands-on experience usin...,"[Microsoft Office Suite, UML, flow charting so...","[UML, DevOps, project management, design, Comp..."
8,Experience with machine learning frameworks (e.g.,[machine learning frameworks],[machine learning]
9,Experience with financial standards like ISO 8...,"[financial standards, ISO 8583, ACH/NACHA]",[]


In [16]:
GENERIC_STOP = {
    "conduct", "be responsible", "responsible",
    "data", "process", "processes",
    "design", "scale",
    "develop", "developing",
    "lead", "leading",
    "driving"
}

def postprocess_spans(spans):
    out = []
    seen = set()
    for s in spans:
        key = norm(s)
        if not key:
            continue
        if key in GENERIC_STOP:
            continue
        if len(key) < 3:
            continue
        if key not in seen:
            seen.add(key)
            out.append(s.strip())
    return out

df["baseline_spans_pp"] = df["baseline_spans_raw"].apply(postprocess_spans)

In [17]:
df[["sentence","gold_spans","baseline_spans_raw","baseline_spans_pp"]].head(10)

,sentence,gold_spans,baseline_spans_raw,baseline_spans_pp
0,Excellent communication and collaboration skil...,[communication and collaboration skills],[communication],[communication]
1,"In this role, you will harness your business a...",[],"[conduct, data, data analysis]",[data analysis]
2,"QualificationsAnalytical Skills, Data Analytic...","[Analytical Skills, Data Analytics, Statistics...","[Data, Data Analytics, communication, statisti...","[Data Analytics, communication, statistical an..."
3,The Business Analyst will be responsible for t...,[],[be responsible],[]
4,"In this role, you will be responsible for desi...",[Drupal-based websites and applications],"[be responsible, Drupal]",[Drupal]
5,Ability to quickly understand health and human...,[],"[business processes, processes]",[business processes]
6,As a Senior Cloud Operations Developer you wil...,"[Cloud solutions, release and deployment proce...","[processes, security, scale]",[security]
7,Minimum five years of hands-on experience usin...,"[Microsoft Office Suite, UML, flow charting so...","[UML, DevOps, project management, design, Comp...","[UML, DevOps, project management, Computer, Co..."
8,Experience with machine learning frameworks (e.g.,[machine learning frameworks],[machine learning],[machine learning]
9,Experience with financial standards like ISO 8...,"[financial standards, ISO 8583, ACH/NACHA]",[],[]


Evaluate baseline with STRICT + LENIENT (phrase-level)

In [18]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def strict_counts(pred_list, gold_list):
    pred = {norm(x) for x in pred_list if str(x).strip()}
    gold = {norm(x) for x in gold_list if str(x).strip()}
    tp = len(pred & gold)
    fp = len(pred - gold)
    fn = len(gold - pred)
    return tp, fp, fn

def lenient_counts(pred_list, gold_list):
    pred = [norm(x) for x in pred_list if str(x).strip()]
    gold = [norm(x) for x in gold_list if str(x).strip()]

    gold_unmatched = set(gold)
    tp = 0
    fp = 0

    for p in pred:
        match = None
        for g in list(gold_unmatched):
            if (p in g) or (g in p):
                match = g
                break
        if match is not None:
            tp += 1
            gold_unmatched.remove(match)
        else:
            fp += 1

    fn = len(gold_unmatched)
    return tp, fp, fn

def prf(tp, fp, fn):
    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    F1 = (2*P*R/(P+R)) if (P+R) else 0.0
    return P, R, F1

In [19]:
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    # if you stored gold spans like "a; b; c"
    return [p.strip() for p in s.split(";") if p.strip()]

df["gold_spans"] = df["gold_spans"].apply(ensure_list)
df["baseline_spans_raw"] = df["baseline_spans_raw"].apply(ensure_list)
df["baseline_spans_pp"]  = df["baseline_spans_pp"].apply(ensure_list)

In [20]:
def evaluate(df_in, pred_col="baseline_spans_raw"):
    s_tp=s_fp=s_fn=0
    l_tp=l_fp=l_fn=0

    for _, r in df_in.iterrows():
        preds = r[pred_col]
        golds = r["gold_spans"]

        tp, fp, fn = strict_counts(preds, golds)
        s_tp += tp; s_fp += fp; s_fn += fn

        tp, fp, fn = lenient_counts(preds, golds)
        l_tp += tp; l_fp += fp; l_fn += fn

    sP, sR, sF1 = prf(s_tp, s_fp, s_fn)
    lP, lR, lF1 = prf(l_tp, l_fp, l_fn)

    return {
        "strict_P": sP, "strict_R": sR, "strict_F1": sF1,
        "lenient_P": lP, "lenient_R": lR, "lenient_F1": lF1,
        "n": len(df_in)
    }

print("=== Baseline (RAW) ===")
raw_metrics = evaluate(df, pred_col="baseline_spans_raw")
print(raw_metrics)

print("\n=== Baseline (POST-PROCESSED) ===")
pp_metrics = evaluate(df, pred_col="baseline_spans_pp")
print(pp_metrics)

=== Baseline (RAW) ===
{'strict_P': 0.17905405405405406, 'strict_R': 0.17055510860820594, 'strict_F1': 0.17470127729707458, 'lenient_P': 0.41469594594594594, 'lenient_R': 0.39501206757843926, 'lenient_F1': 0.4046147507210548, 'n': 700}

=== Baseline (POST-PROCESSED) ===
{'strict_P': 0.23094425483503983, 'strict_R': 0.16331456154465004, 'strict_F1': 0.19132893496701228, 'lenient_P': 0.4869169510807736, 'lenient_R': 0.34432823813354785, 'lenient_F1': 0.40339302544769085, 'n': 700}


In [21]:
def evaluate_by_domain(df_in, pred_col):
    rows = []
    for dom, sub in df_in.groupby("domain"):
        m = evaluate(sub, pred_col=pred_col)
        rows.append({
            "domain": dom,
            "n_sentences": m["n"],
            "strict_P": m["strict_P"], "strict_R": m["strict_R"], "strict_F1": m["strict_F1"],
            "lenient_P": m["lenient_P"], "lenient_R": m["lenient_R"], "lenient_F1": m["lenient_F1"],
        })
    return pd.DataFrame(rows).sort_values("domain")

domain_raw = evaluate_by_domain(df, "baseline_spans_raw")
domain_pp  = evaluate_by_domain(df, "baseline_spans_pp")

display(domain_raw)
display(domain_pp)

domain_raw.to_csv(f"{BASE_DIR}/baseline_raw_metrics_by_domain.csv", index=False)
domain_pp.to_csv(f"{BASE_DIR}/baseline_pp_metrics_by_domain.csv", index=False)

,domain,n_sentences,strict_P,strict_R,strict_F1,lenient_P,lenient_R,lenient_F1
0,DA,165,0.138889,0.188679,0.160000,0.347222,0.471698,0.400000
1,DS,75,0.189655,0.253846,0.217105,0.431034,0.576923,0.493421
2,SWE,460,0.192521,0.154273,0.171288,0.437673,0.350721,0.389402


,domain,n_sentences,strict_P,strict_R,strict_F1,lenient_P,lenient_R,lenient_F1
0,DA,165,0.204082,0.188679,0.196078,0.438776,0.405660,0.421569
1,DS,75,0.248120,0.253846,0.250951,0.496241,0.507692,0.501901
2,SWE,460,0.236364,0.144284,0.179187,0.501818,0.306326,0.380427


Bootstrap 95% CI for F1

In [22]:
def compute_f1_over_df(sample_df, pred_col, mode="strict"):
    tp=fp=fn=0
    for _, r in sample_df.iterrows():
        preds = r[pred_col]
        golds = r["gold_spans"]
        if mode == "strict":
            a,b,c = strict_counts(preds, golds)
        else:
            a,b,c = lenient_counts(preds, golds)
        tp += a; fp += b; fn += c
    return prf(tp, fp, fn)[2]

def bootstrap_ci(df_in, pred_col, mode="strict", n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    f1s = []
    n = len(df_in)
    idx = np.arange(n)
    for _ in range(n_boot):
        samp_idx = rng.choice(idx, size=n, replace=True)
        samp = df_in.iloc[samp_idx]
        f1s.append(compute_f1_over_df(samp, pred_col=pred_col, mode=mode))
    f1s = np.array(f1s)
    return float(np.percentile(f1s, 2.5)), float(np.percentile(f1s, 97.5))

In [24]:
raw_strict_ci = bootstrap_ci(df, "baseline_spans_raw", mode="strict")
raw_len_ci    = bootstrap_ci(df, "baseline_spans_raw", mode="lenient")

pp_strict_ci  = bootstrap_ci(df, "baseline_spans_pp", mode="strict")
pp_len_ci     = bootstrap_ci(df, "baseline_spans_pp", mode="lenient")

print("RAW strict CI:", raw_strict_ci, " RAW lenient CI:", raw_len_ci)
print("PP  strict CI:", pp_strict_ci,  " PP lenient CI:", pp_len_ci)

RAW strict CI: (0.15174103611922576, 0.1994610239405645)  RAW lenient CI: (0.3756457154503897, 0.43016449918656263)
PP  strict CI: (0.16462762765412928, 0.21805579668531896)  PP lenient CI: (0.3716811803179809, 0.43219232142443903)


In [25]:
ci_df = pd.DataFrame([
    {
        "model": "baseline_raw",
        "strict_f1": raw_metrics["strict_F1"],
        "strict_ci_low": raw_strict_ci[0],
        "strict_ci_high": raw_strict_ci[1],
        "lenient_f1": raw_metrics["lenient_F1"],
        "lenient_ci_low": raw_len_ci[0],
        "lenient_ci_high": raw_len_ci[1],
        "n_sentences": len(df)
    },
    {
        "model": "baseline_postprocessed",
        "strict_f1": pp_metrics["strict_F1"],
        "strict_ci_low": pp_strict_ci[0],
        "strict_ci_high": pp_strict_ci[1],
        "lenient_f1": pp_metrics["lenient_F1"],
        "lenient_ci_low": pp_len_ci[0],
        "lenient_ci_high": pp_len_ci[1],
        "n_sentences": len(df)
    }
])

CI_PATH = f"{BASE_DIR}/baseline_f1_with_bootstrap_ci.csv"
ci_df.to_csv(CI_PATH, index=False)
CI_PATH

'/content/drive/MyDrive/CS685/linkedin/baseline_f1_with_bootstrap_ci.csv'

In [26]:
OUT_PRED_PATH = f"{BASE_DIR}/baseline_predictions_with_gold_raw_and_pp.csv"
df.to_csv(OUT_PRED_PATH, index=False)
OUT_PRED_PATH

'/content/drive/MyDrive/CS685/linkedin/baseline_predictions_with_gold_raw_and_pp.csv'